# Figure S5: coefficient magnitude against structural distance

First-order coefficient magnitudes plotted against the minimum heavy-atom distance between each mutated residue and the antigen, and pairwise magnitudes against the Ca-Ca distance between mutated residues, for the five antibodies with a somatic Fab-gp120 co-crystal structure. Spearman correlations are reported per antigen and corrected across antibodies by Benjamini-Hochberg.

All five structures are antibody bound to a clade A/E 93TH057 gp120 core rather than to CH505TF or SF162, so the same coordinates serve both antigen-specific coefficient sets.

## 1. Setup

In [ ]:
# Biopython parses the mmCIF structures. Not preinstalled on Colab.
!pip install -q biopython

In [ ]:
import re
import urllib.request
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=Warning, module="Bio")
%matplotlib inline

## 2. Paths and constants

folder. Structures are cached under `STRUCTURE_DIR`, which is local to the
Colab VM - they re-download on a fresh runtime, which takes a few seconds.

In [ ]:
import os

data_dir = "../data"
fig_dir = "../figures"
table_dir = "../tables"
structure_dir = f"{data_dir}/structures"   # mmCIF cache, committed so the figure reruns offline
for d in (fig_dir, table_dir, structure_dir):
    os.makedirs(d, exist_ok=True)
PNG_DPI = 300

plt.rcParams["svg.fonttype"] = "path"   # illustrator drops live text
plt.rcParams["pdf.fonttype"] = 42

def save_figure(fig, stem):
    """Write png, pdf and svg, then check the svg has no live text."""
    path = f"{fig_dir}/{stem}"
    fig.savefig(f"{path}.png", dpi=PNG_DPI, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", bbox_inches="tight")
    fig.savefig(f"{path}.svg", bbox_inches="tight")
    assert open(f"{path}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"


# Only these five antibodies have a resolved somatic co-crystal structure AND
# a Tobit fit, so only they can appear in a distance plot. Order here is the
# left-to-right column order in the figure.
ANTIBODIES = ["CH235.12", "VRC13", "VRC01", "VRC07", "N49P7"]
ANTIGENS = ("SF162", "CH505TF")

# Coefficient filenames use a sanitised antibody stem ('.' and '-' removed).
FINAL_FILE_STEM = {
    "CH235.12": "CH235_12", "VRC13": "VRC13", "VRC01": "VRC01",
    "VRC07": "VRC07", "N49P7": "N49P7",
}

# Somatic co-crystal structure per antibody, with chain assignments.
# Chain conventions were verified directly against each structure:
#   G = antigen (gp120), H = antibody heavy chain, L = antibody light chain.
# VRC07 note: 4OLU is one entry in an 8-structure position-54 saturation
# series and carries Gly at 54, NOT the S54F mutation in our binding data.
# 4OLV carries Phe and matches all 10 of VRC07's sites; it is used instead.
STRUCTURES = {
    "CH235.12": {"pdb_id": "5f96", "heavy_chain": "H", "light_chain": "L", "antigen_chains": ["G"]},
    "VRC13":    {"pdb_id": "4ydj", "heavy_chain": "H", "light_chain": "L", "antigen_chains": ["G"]},
    "VRC01":    {"pdb_id": "3ngb", "heavy_chain": "H", "light_chain": "L", "antigen_chains": ["G"]},
    "VRC07":    {"pdb_id": "4olv", "heavy_chain": "H", "light_chain": "L", "antigen_chains": ["G"]},
    "N49P7":    {"pdb_id": "6bck", "heavy_chain": "H", "light_chain": "L", "antigen_chains": ["G"]},
}

AA3TO1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V", "PCA": "Q",
}

## 3. Loading structures

mmCIF files are fetched from RCSB on first use and cached on disk; parsed
models are cached in memory so each structure is only parsed once.

In [ ]:
_model_cache = {}


def fetch_cif(pdb_id):
    # Download the mmCIF from RCSB if it isn't cached yet. Returns the path.
    path = Path(structure_dir) / f"{pdb_id}.cif"
    if not path.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.cif"
        print(f"  downloading {pdb_id}.cif ...")
        urllib.request.urlretrieve(url, path)
    return path


def load_model(pdb_id):
    # Parse (and cache) the first model of a structure.
    if pdb_id not in _model_cache:
        parser = MMCIFParser(QUIET=True)
        structure = parser.get_structure(pdb_id, str(fetch_cif(pdb_id)))
        _model_cache[pdb_id] = list(structure)[0]
    return _model_cache[pdb_id]

## 4. Site lookup: Kabat position → residue

Kabat numbering in these PDB files already uses insertion codes (e.g. `52A`)
the same way our `kabat_position` column does, so the lookup is a direct
`(resseq, icode)` match - no sequence alignment or interpolation needed.

In [ ]:
def parse_kabat_position(pos):
    # '52A' -> (52, 'A');  '35' -> (35, '')
    m = re.match(r"^(\d+)([A-Za-z]*)$", str(pos).strip())
    if not m:
        raise ValueError(f"Cannot parse Kabat position: {pos!r}")
    return int(m.group(1)), m.group(2).upper()


def get_residue(model, chain_id, resseq, icode=""):
    # Find a residue by number + insertion code within one chain.
    icode = icode if icode else " "
    if chain_id not in model:
        return None
    for res in model[chain_id]:
        if not is_aa(res, standard=False):
            continue
        if int(res.id[1]) == int(resseq) and (res.id[2].strip() or " ") == icode:
            return res
    return None


def load_meta(antibody):
    # Site metadata: one row per mutated site, with Kabat numbering.
    meta = pd.read_csv(f"{data_dir}/{FINAL_FILE_STEM[antibody]}_meta.csv")
    meta.columns = meta.columns.str.strip()
    meta["site_index"] = meta["site"].astype(str).str.replace("site_", "", regex=False).astype(int)
    # Compact mutation label, e.g. 'Q3R' = germline Q at Kabat 3 -> mature R
    meta["kabat"] = (meta["germline_aa"].astype(str)
                     + meta["kabat_position"].astype(str)
                     + meta["mature_aa"].astype(str))
    return meta.sort_values("site_index").reset_index(drop=True)

## 5. Distance geometry

Two distances are computed:

- **Site → antigen** (Figure 3C): the minimum distance between *any* heavy
  atom of the mutated residue and *any* heavy atom of *any* antigen residue.
  This is the closest-approach definition, not a centroid proxy.
- **Site ↔ site** (Figure 3D): the Cα–Cα distance between two mutated
  residues - the standard backbone proxy for inter-residue separation.

Hydrogens are excluded throughout (they are usually unresolved in
crystal structures, so including them would be inconsistent between entries).

In [ ]:
def heavy_atoms(residue):
    # Non-hydrogen atoms only.
    return [a for a in residue.get_atoms()
            if a.element != "H" and not a.get_id().startswith("H")]


def residue_coords(residue):
    atoms = heavy_atoms(residue)
    return np.array([a.coord for a in atoms], dtype=float) if atoms else np.empty((0, 3))


def ca_coord(residue):
    # Cα coordinate, falling back to the heavy-atom centroid if Cα is missing.
    if "CA" in residue:
        return residue["CA"].coord.astype(float)
    xyz = residue_coords(residue)
    return xyz.mean(axis=0) if xyz.shape[0] else np.array([np.nan, np.nan, np.nan])


def min_distance_to_antigen(model, residue, antigen_chains):
    # Closest approach between this residue and any antigen residue (A).
    x = residue_coords(residue)
    if x.shape[0] == 0:
        return np.nan
    best = np.inf
    for chain_id in antigen_chains:
        if chain_id not in model:
            continue
        for ag_res in model[chain_id]:
            if not is_aa(ag_res, standard=False):
                continue
            y = residue_coords(ag_res)
            if y.shape[0] == 0:
                continue
            # pairwise atom-atom distances, keep the smallest
            d = np.sqrt(((x[:, None, :] - y[None, :, :]) ** 2).sum(axis=2)).min()
            best = min(best, float(d))
    return best if np.isfinite(best) else np.nan

In [ ]:
def build_site_structural_table(antibody, verify=True):
    # One row per mutated site: Kabat label, region, and distance to antigen.
    cfg = STRUCTURES[antibody]
    model = load_model(cfg["pdb_id"])
    meta = load_meta(antibody)

    rows = []
    for _, r in meta.iterrows():
        base, icode = parse_kabat_position(r["kabat_position"])
        res = get_residue(model, cfg["heavy_chain"], base, icode)
        row = {
            "antibody": antibody, "pdb_id": cfg["pdb_id"],
            "site_index": int(r["site_index"]), "kabat": r["kabat"],
            "kabat_position": r["kabat_position"], "region": r.get("region"),
            "mature_aa": r["mature_aa"], "found": res is not None,
        }
        if res is None:
            row.update({"ca": None, "min_dist_to_antigen_A": np.nan, "struct_aa": None})
        else:
            row.update({
                "ca": ca_coord(res),
                "min_dist_to_antigen_A": min_distance_to_antigen(model, res, cfg["antigen_chains"]),
                "struct_aa": AA3TO1.get(res.get_resname().upper()),
            })
        rows.append(row)

    table = pd.DataFrame(rows)

    # QC: the residue found in the structure should match the mature amino
    # acid in our metadata. A mismatch means the wrong PDB entry or chain --
    # which is exactly how the VRC07 4OLU/4OLV problem was caught.
    if verify:
        bad = table[table["found"] & (table["struct_aa"] != table["mature_aa"])]
        if len(bad):
            print(f"  !! {antibody}: {len(bad)} site(s) disagree with the structure:")
            print(bad[["kabat", "mature_aa", "struct_aa"]].to_string(index=False))
    return table


def build_pairwise_ca_table(site_table):
    # Cα-Cα distance for every unordered pair of mutated sites.
    d = site_table[site_table["found"]]
    rows = []
    for _, a in d.iterrows():
        for _, b in d.iterrows():
            if a["site_index"] >= b["site_index"]:
                continue
            rows.append({
                "site_i": a["site_index"], "site_j": b["site_index"],
                "pair": f"{a['kabat']}:{b['kabat']}",
                "ca_distance_A": float(np.linalg.norm(a["ca"] - b["ca"])),
            })
    return pd.DataFrame(rows)

## 6. Coefficients

The Tobit fits are precomputed. Terms are named `site_<i>` for first-order
and `site_<i>,site_<j>` for pairwise, so the site indices parse straight out
of the term string and join to the structural table above.

In [ ]:
def find_final_params_file(antibody, antigen):
    # Best-fit coefficients for one complex, or None if it was not fitted.
    fp = f"{data_dir}/{FINAL_FILE_STEM[antibody]}_{antigen}_BEST_df_params.csv"
    return fp if os.path.exists(fp) else None


def load_params(antibody, antigen):
    # Load one Tobit fit and parse each term into its list of site indices.
    fp = find_final_params_file(antibody, antigen)
    if fp is None:
        raise FileNotFoundError(f"No coefficient file for {antibody} {antigen}")
    params = pd.read_csv(fp)
    params.columns = params.columns.str.strip()
    params["coefficient"] = pd.to_numeric(params["coefficient"], errors="coerce")
    params["site_list"] = params["term"].apply(
        lambda t: [int(x.split("_")[1]) for x in str(t).split(",") if x.strip().startswith("site_")]
    )
    return params

In [ ]:
def build_first_order_table(antibody):
    # First-order coefficients joined to each site's distance to antigen.
    site_table = build_site_structural_table(antibody)
    dist_by_site = dict(zip(site_table["site_index"], site_table["min_dist_to_antigen_A"]))
    region_by_site = dict(zip(site_table["site_index"], site_table["region"]))
    kabat_by_site = dict(zip(site_table["site_index"], site_table["kabat"]))

    rows = []
    for antigen in ANTIGENS:
        params = load_params(antibody, antigen)
        first = params[params["order"] == 1].copy()
        first["site_index"] = first["site_list"].apply(lambda s: s[0])
        first["dist"] = first["site_index"].map(dist_by_site)
        first["region"] = first["site_index"].map(region_by_site)
        first["label"] = first["site_index"].map(kabat_by_site)
        first["antigen"] = antigen
        rows.append(first[["site_index", "coefficient", "dist", "region", "label", "antigen"]])

    d = pd.concat(rows, ignore_index=True)
    d = d[d["dist"].notna()].copy()
    d["abs_coefficient"] = d["coefficient"].abs()
    return d


def build_pairwise_table(antibody):
    # Pairwise coefficients joined to each pair's Ca-Ca distance.
    site_table = build_site_structural_table(antibody)
    pair_dist = build_pairwise_ca_table(site_table)
    dist_by_pair = {(r.site_i, r.site_j): (r.ca_distance_A, r.pair)
                    for r in pair_dist.itertuples()}

    rows = []
    for antigen in ANTIGENS:
        params = load_params(antibody, antigen)
        for _, r in params[params["order"] == 2].iterrows():
            i, j = r["site_list"]
            key = (i, j) if (i, j) in dist_by_pair else (j, i)
            if key not in dist_by_pair:
                continue
            dist, label = dist_by_pair[key]
            rows.append({"coefficient": r["coefficient"], "dist": dist,
                         "label": label, "antigen": antigen})

    d = pd.DataFrame(rows)
    d["abs_coefficient"] = d["coefficient"].abs()
    return d

## 7. Statistics

Spearman ρ is computed **within each antigen** - SF162 and CH505TF are
separate measurements and are never pooled into one correlation.

The BH-FDR family is **one panel**: all five antibodies × both antigens
(m = 10). First-order and pairwise are corrected separately, because they are
different quantities measured at very different sample sizes (n = 10 sites vs.
n = 45 pairs), so they are not a single scan.

In [ ]:
def spearman_by_antigen(df):
    # {antigen: (rho, p, n)} for one antibody's table.
    out = {}
    for antigen in ANTIGENS:
        sub = df[df["antigen"] == antigen]
        x = sub["dist"].to_numpy(float)
        y = sub["abs_coefficient"].to_numpy(float)
        m = (~np.isnan(x)) & (~np.isnan(y))
        if m.sum() < 3:
            out[antigen] = (np.nan, np.nan, int(m.sum()))
        else:
            rho, p = spearmanr(x[m], y[m])
            out[antigen] = (rho, p, int(m.sum()))
    return out


def compute_panel_stats(tables):
    # tables: {antibody: dataframe} for ONE panel (first-order or pairwise).
    # Returns a tidy frame with BH-FDR q over all 10 tests in that panel.
    recs = []
    for ab, d in tables.items():
        for antigen, (rho, p, n) in spearman_by_antigen(d).items():
            recs.append({"antibody": ab, "antigen": antigen, "n": n, "rho": rho, "p": p})
    stats = pd.DataFrame(recs)
    _, q, _, _ = multipletests(stats["p"].to_numpy(), alpha=0.05, method="fdr_bh")
    stats["q"] = q
    return stats

## 8. Plot styling

Marker **shape** encodes antigen; marker **colour** encodes the sign of the
original (signed) coefficient, since the y-axis shows magnitude and would
otherwise discard that information.

Both axes are shared across all five antibodies within a panel, so magnitudes
are directly comparable along the row. The y-axis stops just above the row's
own maximum rather than at the next round multiple of the tick step - a
coefficient range of 0.53 would otherwise round up to an axis of 1.0 and waste
half the panel height.

In [ ]:
ANTIGEN_MARKER = {"SF162": "s", "CH505TF": "^"}
# Marker area in points^2, tuned for the ~6 inch wide subplots produced below.
# The manuscript render uses much larger panels (~10.8 in wide) and scales
# these up to 600/700 accordingly -- marker size in points does NOT follow the
# figure size automatically, so it has to be retuned whenever figsize changes.
ANTIGEN_MARKER_SIZE = {"SF162": 180, "CH505TF": 210}
ANTIGEN_RHO_SUBSCRIPT = {"SF162": "□", "CH505TF": "△"}  # hollow square / triangle

POS_COLOR, NEG_COLOR = "#C0392B", "#2980B9"   # coefficient sign
ZERO_LINE_COLOR = "#006400"                   # dark green y=0 reference

X_TICK_COUNT = 5
_NICE_Y_STEPS = [1, 2, 2.5, 4, 5, 10]


def _tight_y_step(ymax, want_ticks=6, lo=4, hi=8):
    # Nice-rounded y step that puts a sensible NUMBER of ticks at or below
    # ymax, so the axis can stop just past the data instead of at the next
    # whole multiple of the step.
    if not np.isfinite(ymax) or ymax <= 0:
        return 1.0
    best, best_cost = None, None
    for mag_exp in range(-6, 7):
        for m in _NICE_Y_STEPS:
            step = m * (10.0 ** mag_exp)
            n = int(np.floor(ymax / step + 1e-9)) + 1
            if n < lo or n > hi:
                continue
            cost = (abs(n - want_ticks), -n)
            if best_cost is None or cost < best_cost:
                best, best_cost = step, cost
    return best if best is not None else ymax / (want_ticks - 1)


def _nice_x_step(raw_step):
    # Round a raw step up to the nearest integer multiple of its magnitude
    # (1..10 x 10^k). Denser buckets than the y axis so a panel's x range
    # stays close to its own data.
    if not np.isfinite(raw_step) or raw_step <= 0:
        return 1.0
    magnitude = 10 ** np.floor(np.log10(raw_step))
    for m in range(1, 11):
        step = m * magnitude
        if step >= raw_step - 1e-15:
            return step
    return 10 * magnitude

In [ ]:
def draw_panel(ax, df, stats_row, xmax, ymax, x_margin=1.0, show_yticks=True):
    # One antibody's scatter, with the rho/q annotation in the top-right.
    for antigen in ANTIGENS:
        sub = df[df["antigen"] == antigen]
        if sub.empty:
            continue
        colors = [POS_COLOR if c >= 0 else NEG_COLOR for c in sub["coefficient"]]
        ax.scatter(sub["dist"], sub["coefficient"].abs(),
                   marker=ANTIGEN_MARKER[antigen], s=ANTIGEN_MARKER_SIZE[antigen],
                   c=colors, edgecolor="black", linewidth=1.1, alpha=0.55, zorder=3)

    # --- y axis: shared across the row, tight to the row's max ---
    step = _tight_y_step(ymax)
    n_ticks = int(np.floor(ymax / step + 1e-9)) + 1
    ax.set_ylim(-step * 0.6, ymax * 1.04)   # headroom below 0 so zero points read
    ax.set_yticks([i * step for i in range(n_ticks)])

    # --- x axis: shared across the row, fixed tick count ---
    xstep = _nice_x_step(xmax / (X_TICK_COUNT - 1))
    top = xstep * (X_TICK_COUNT - 1)
    ax.set_xlim(-xstep * 0.05, top * x_margin)
    ax.set_xticks([i * xstep for i in range(X_TICK_COUNT)])

    ax.axhline(0, color=ZERO_LINE_COLOR, linestyle="--", linewidth=1.6, zorder=2)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(axis="both", labelsize=13)
    if not show_yticks:
        ax.tick_params(labelleft=False)

    # rho and FDR-corrected q, one line per antigen
    lines = []
    for antigen in ANTIGENS:
        r = stats_row[stats_row["antigen"] == antigen].iloc[0]
        g = ANTIGEN_RHO_SUBSCRIPT[antigen]
        lines.append(rf"$\rho_{{\mathrm{{{g}}}}} = {r['rho']:.2f}$, $q = {r['q']:.3f}$")
    ax.text(0.96, 0.95, "\n".join(lines), transform=ax.transAxes,
            fontsize=13, ha="right", va="top", zorder=6,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2))

In [ ]:
def plot_distance_row(term_type, out_name=None, show=True, annotate_q=True):
    # term_type: 'first' (Figure S5A) or 'pair' (Figure S5B).
    builder = build_first_order_table if term_type == "first" else build_pairwise_table
    xlabel = ("Min. distance to antigen, Å" if term_type == "first"
              else "Cα-Cα distance, Å")
    ylabel = (r"|Model first-order coefficient|" if term_type == "first"
              else r"|Model pairwise coefficient|")
    # The pairwise cloud bunches up, so its x axis gets a little extra room.
    x_margin = 1.0 if term_type == "first" else 1.05

    print(f"Building {term_type} tables ...")
    tables = {ab: builder(ab) for ab in ANTIBODIES}
    stats = compute_panel_stats(tables)

    # Shared limits across the whole row.
    xmax = max(t["dist"].max() for t in tables.values())
    ymax = max(t["abs_coefficient"].max() for t in tables.values())

    fig, axes = plt.subplots(1, len(ANTIBODIES), figsize=(30, 5.6), dpi=120)
    for k, (ax, ab) in enumerate(zip(axes, ANTIBODIES)):
        draw_panel(ax, tables[ab], stats[stats["antibody"] == ab],
                   xmax, ymax, x_margin=x_margin, show_yticks=True)
        ax.set_title(ab, fontsize=20, fontweight="bold", pad=12)
        ax.set_xlabel(xlabel, fontsize=15)
    axes[0].set_ylabel(ylabel, fontsize=15)
    fig.tight_layout()

    if out_name:
        save_figure(fig, Path(out_name).stem)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return stats

## 9. Make the figures

Each call downloads any missing structures, computes the distances, joins the
coefficients, runs the statistics, and draws the panel. The returned frame is
the per-antibody statistics for that panel.

In [ ]:
# Figure 3C -- first-order coefficients vs. distance to antigen
stats_C = plot_distance_row("first", out_name="figure3C_first_order.png")
stats_C

In [ ]:
# Figure 3D -- pairwise coefficients vs. Ca-Ca distance
stats_D = plot_distance_row("pair", out_name="figure3D_pairwise.png")
stats_D

## 10. Supplementary statistics table

Combines both panels into the table reported in the supplement. `q` is the
BH-FDR value within that panel (m = 10).

In [ ]:
stats_table = pd.concat([
    stats_C.assign(term_type="First-order"),
    stats_D.assign(term_type="Pairwise"),
], ignore_index=True)[["term_type", "antibody", "antigen", "n", "rho", "p", "q"]]

stats_table.to_csv(f"{table_dir}/distance_stats_supplementary_table.csv", index=False)
print(f"significant at q < 0.05: {(stats_table['q'] < 0.05).sum()} / {len(stats_table)}")
stats_table.round(4)